# Orbital symmetry blocks and the winding number

When the active orbitals fall into irreducible representations that do not mix, $G$ is
block diagonal and $\det G$ factorises. Each block then carries its own **winding number**:
the number of times $\det G(z)$ circles the origin as $z$ traverses a closed contour. By the
argument principle that integer is

$$
\text{winding} = (\text{zeros of } \det G \text{ inside}) - (\text{poles inside})
$$

and an integer cannot change continuously — it can only jump when a pole and a zero
collide.

This notebook works through the machinery on rectangular H4, which PySCF assigns to D2h, so
the blocking is exact rather than approximate.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from pyscf import gto

from casgf import (
    ActiveSpace,
    block_leakage,
    det_along_contour,
    irrep_blocks,
    keyhole_contour,
    lehmann,
    min_pole_distance,
    winding_number,
    winding_number_of,
)

CONTOUR = keyhole_contour(radius=1.5, n_per_segment=10_000)

In [ ]:
def benzene(cc=1.39, ch=1.09):
    """Idealised D6h benzene: a regular hexagon of carbons with radial C-H bonds."""
    atoms = []
    for k in range(6):
        c, s = np.cos(2 * np.pi * k / 6), np.sin(2 * np.pi * k / 6)
        atoms.append(f"C {cc * c:.8f} {cc * s:.8f} 0.0")
        atoms.append(f"H {(cc + ch) * c:.8f} {(cc + ch) * s:.8f} 0.0")
    return "; ".join(atoms)


def h4(width, height=2.6):
    """Four hydrogens on the corners of a rectangle, in Bohr.

    A compact strongly correlated test case: four 1s orbitals, four electrons,
    and a real point group (D2h, rising to D4h when width == height).
    """
    x, y = width / 2, height / 2
    return "; ".join(
        f"H {sx * x:.8f} {sy * y:.8f} 0.0"
        for sx, sy in ((1, 1), (-1, 1), (-1, -1), (1, -1))
    )

## The active space, with symmetry switched on

`mol.symmetry = True` is what makes the blocking meaningful: it asks PySCF to classify the
orbitals, and `ActiveSpace` carries those labels along.

In [ ]:
mol = gto.M(atom=h4(2.0), basis="6-31g", unit="B", symmetry=True, verbose=0)
space = ActiveSpace.from_molecule(mol, ncas=4, nelecas=4)
gf = lehmann(space)

blocks = irrep_blocks(space.orbsym, mol)
print(f"point group      {mol.groupname}")
print(f"active orbitals  { {name: idx.tolist() for name, idx in blocks.items()} }")
print(f"mu {gf.mu:+.6f}   gap {gf.gap:.6f}   sum rule {gf.sum_rule():.9f}")

## How exact is the blocking?

Worth measuring rather than assuming. `block_leakage` returns the largest
$|G_{ij}| / \sqrt{|G_{ii}| |G_{jj}|}$ connecting different blocks — zero exactly when $G$ is
block diagonal and $\det G$ factorises exactly.

In [ ]:
leak = block_leakage(gf.at(0.0, eta=0.05), blocks.values())
print(f"block leakage {leak:.1e}")

# Factorisation, checked directly on the determinant.
freqs = np.linspace(-1, 1, 201)
full = gf.det(freqs, eta=0.05)
product = np.ones_like(full)
for idx in blocks.values():
    product *= gf.det(freqs, eta=0.05, orbitals=idx)
print(f"max |det G - prod(det G_block)| = {np.abs(full - product).max():.1e}")

## Winding numbers

The contour is the boundary of the right half-disc of radius 1.5, traversed anticlockwise.
Check the clearance first: a contour that grazes a pole makes $\arg\det G$ jump by more than
$\pi$ between samples, which would corrupt the phase unwrapping.

In [ ]:
print(f"closest pole to the contour: {min_pole_distance(gf, CONTOUR):.2e}")

total = winding_number_of(gf, CONTOUR)
per_block = {name: winding_number_of(gf, CONTOUR, orbitals=idx) for name, idx in blocks.items()}
for name, value in per_block.items():
    print(f"  {name:4s} {value:+d}")
print(f"  {'sum':4s} {sum(per_block.values()):+d}    (whole active space: {total:+d})")

In [ ]:
def plot_traces(gf, blocks, contour, title):
    names = list(blocks)
    fig, axes = plt.subplots(1, len(names), figsize=(3.2 * len(names), 3.4), dpi=120)
    colour = np.linspace(0, 1, len(contour))
    for ax, name in zip(np.atleast_1d(axes), names, strict=True):
        values = det_along_contour(gf, contour, orbitals=blocks[name])
        ax.scatter(values.real, values.imag, s=0.2, c=colour, cmap=cm.gnuplot)
        ax.axhline(0, color="k", lw=0.5, ls="--")
        ax.axvline(0, color="k", lw=0.5, ls="--")
        ax.set_title(f"{name}   ({winding_number(values):+d})", fontsize=10)
        ax.set_xlabel(r"$\mathrm{Re}\,\det G$")
    np.atleast_1d(axes)[0].set_ylabel(r"$\mathrm{Im}\,\det G$")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_traces(gf, blocks, CONTOUR, "H4 rectangle, CAS(4,4)/6-31G — colour runs along the contour")

## Where the zeros show up

For a non-interacting Green's function $\det G$ has no zeros at all, so the winding number
just counts poles. Comparing the two on the same contour therefore isolates the
contribution of the zeros — which is the whole reason $\log|\det G|$ is worth looking at.

In [ ]:
free = lehmann(ActiveSpace.from_arrays(space.h1, np.zeros_like(space.eri), space.nelecas))

print(f"{'':16s} {'winding':>8} {'poles in':>9} {'zeros in':>9}")
for label, g in (("interacting", gf), ("non-interacting", free)):
    # Only poles carrying spectral weight are poles of G at all: most of the
    # N+-1 spectrum is orthogonal to c^dag|0> and drops out of the residues.
    weights = g.pole_weights()
    poles_in = int(np.count_nonzero((g.poles > 0) & (g.poles < 1.5) & (weights > 1e-10)))
    winding = winding_number_of(g, CONTOUR)
    print(f"{label:16s} {winding:+8d} {poles_in:9d} {winding + poles_in:9d}")

The non-interacting row has **exactly zero** enclosed zeros, which is what it has to be:
without the interaction $\det G$ is a product of $1/(z - \varepsilon_k)$ and cannot vanish.
Every zero counted in the interacting row is therefore produced by the interaction — and
those are precisely the features that are invisible in the spectral function.

## Notes

- The origin lies *on* the contour, on its imaginary-axis segment. That is harmless here:
  the chemical potential puts $\omega = 0$ in the middle of the gap, where $\det G$ is finite
  and non-zero. `winding_number` raises if $\det G$ ever vanishes exactly on the contour.
- Blocks can also be selected as `"even"` / `"odd"` — index parity, `G[::2, ::2]` and
  `G[1::2, 1::2]`. That is a fallback for when point-group symmetry is unavailable, and it
  is only as good as the assumption that the orbitals alternate between two irreps.
  `block_leakage` is how you find out whether it holds.